In [1]:
import os
import re
import pandas as pd


def parse_applicant_name(raw_name: str) -> str:
    """Cleans salutations, handles missing/NA values, extracts first name or

    first+middle name based on rules, skips 1-2 character initials/short forms
    to use middle/last names when available, and converts result to Title Case.
    """
    if pd.isna(raw_name) or not isinstance(raw_name, str):
        return "Sir"

    name_str = raw_name.strip()

    # Handle explicit missing / empty indicators
    if name_str.lower() in ["", "n/a", "na", "n.a.", "null", "none", "nan"]:
        return "Sir"

    # Remove common salutations (e.g., Mr., Mrs., Ms., Dr., Prof., Rev., Er., Mx.)
    salutation_pattern = r"^(?:(mr|mrs|ms|dr|prof|rev|er|mx)\.?\s+)+"
    name_str = re.sub(salutation_pattern, "", name_str, flags=re.IGNORECASE).strip()

    # Re-check after removing salutations
    if name_str.lower() in ["", "n/a", "na", "n.a.", "null", "none", "nan"]:
        return "Sir"

    tokens = [token for token in name_str.split() if token]
    if not tokens:
        return "Sir"

    # Clean punctuation/dots from tokens (e.g., "B." -> "B", "Rk." -> "Rk")
    clean_tokens = [re.sub(r"^[^\w]+|[^\w]+$", "", t) for t in tokens]
    clean_tokens = [t for t in clean_tokens if t]

    if not clean_tokens:
        return "Sir"

    # Filter out 1-2 character initials/abbreviations (e.g., B, Rk, B., Md, Ch)
    long_tokens = [t for t in clean_tokens if len(t) > 2]

    # Use full name tokens if available; fallback to clean tokens if all are short
    working_tokens = long_tokens if long_tokens else clean_tokens

    num_tokens = len(working_tokens)

    if num_tokens == 1:
        processed_name = working_tokens[0]
    elif num_tokens == 2:
        # First name + Last name (or remaining) -> keep First name only
        processed_name = working_tokens[0]
    else:
        # First, Middle, + Last name (3+) -> keep First and Middle name
        processed_name = f"{working_tokens[0]} {working_tokens[1]}"

    return processed_name.title()


def generate_emailing_list():
    # Read file paths from environment variables
    input_file_path = os.environ.get("ActiveLeads")
    output_file_path = os.environ.get("email")

    if not input_file_path:
        raise ValueError(
            "Environment variable 'ActiveLeads' is not defined or empty."
        )
    if not output_file_path:
        raise ValueError("Environment variable 'email' is not defined or empty.")

    # Ensure output file ends with .xlsx extension
    if not output_file_path.lower().endswith(".xlsx"):
        output_file_path += ".xlsx"

    # Read input file (supports both CSV and Excel)
    if input_file_path.lower().endswith(".csv"):
        df = pd.read_csv(input_file_path)
    else:
        df = pd.read_excel(input_file_path)

    # Ensure required columns exist
    required_cols = {"Email_1", "Email_2", "Applicant_Name"}
    if not required_cols.issubset(df.columns):
        missing = required_cols - set(df.columns)
        raise KeyError(f"Input file is missing required columns: {missing}")

    # Build entry lists for Email_1 and Email_2
    list_1 = df[["Email_1", "Applicant_Name"]].rename(columns={"Email_1": "Email"})
    list_2 = df[["Email_2", "Applicant_Name"]].rename(columns={"Email_2": "Email"})

    # Combine both email sources into a single dataframe
    combined_df = pd.concat([list_1, list_2], ignore_index=True)

    # Filter out missing/blank emails
    combined_df = combined_df.dropna(subset=["Email"])
    combined_df["Email"] = combined_df["Email"].astype(str).str.strip()
    combined_df = combined_df[combined_df["Email"] != ""]

    # Transform names using updated parsing logic
    combined_df["Name"] = combined_df["Applicant_Name"].apply(
        parse_applicant_name
    )

    # Format final columns and drop duplicate emails
    final_email_list = combined_df[["Name", "Email"]].drop_duplicates(
        subset=["Email"], keep="first"
    )

    # Export to Excel
    final_email_list.to_excel(output_file_path, index=False, engine="openpyxl")
    print(
        f"Email list successfully exported to {output_file_path} ({len(final_email_list)} unique entries)."
    )


if __name__ == "__main__":
    generate_emailing_list()

Email list successfully exported to F:\Chimney Work\Marketing\LeadGen\Data Architecture\3 - Gold\Email list\Email list.xlsx (3775 unique entries).


In [2]:
#Add the master email list

import os
import pandas as pd


def update_email_database():
    # Retrieve file paths from environment variables
    target_path = os.getenv("emailMDB")
    source_path = os.getenv("email")

    if not target_path or not source_path:
        raise ValueError(
            "Environment variables 'emailMDB' and 'email' must be set."
        )

    # Read both Excel files
    # Note: Requires 'openpyxl' installed (pip install pandas openpyxl)
    df_target = pd.read_excel(target_path)
    df_source = pd.read_excel(source_path)

    # Ensure required columns exist
    required_cols = ["Email", "Name"]
    for col in required_cols:
        if col not in df_target.columns or col not in df_source.columns:
            raise KeyError(
                f"Both files must contain '{col}' column. Found target: {list(df_target.columns)}, source: {list(df_source.columns)}"
            )

    # Standardize email strings for accurate deduplication (lowercase & strip whitespace)
    existing_emails = set(
        df_target["Email"].dropna().astype(str).str.strip().str.lower()
    )

    # Filter source rows where Email is valid and not already in target
    source_emails_clean = (
        df_source["Email"].dropna().astype(str).str.strip().str.lower()
    )
    new_rows_mask = ~source_emails_clean.isin(existing_emails)

    # Extract only the unique new rows and keep required columns
    df_new = df_source[new_rows_mask][required_cols].drop_duplicates(
        subset=["Email"]
    )

    if df_new.empty:
        print("No new email IDs to add.")
        return

    # Combine original target data with new rows and save back to target file
    df_updated = pd.concat([df_target, df_new], ignore_index=True)
    df_updated.to_excel(target_path, index=False)

    print(f"Successfully added {len(df_new)} new record(s) to '{target_path}'.")
    


if __name__ == "__main__":
    update_email_database()

os.remove(os.getenv("email"))

Successfully added 2 new record(s) to 'F:\Chimney Work\Marketing\LeadGen\Data Architecture\3 - Gold\Email list\EmailMasterDB.xlsx'.


In [3]:
#create batch file for today's send
import os
from datetime import datetime, timedelta
import pandas as pd

# 1. Fetch file paths from environment variables
mdb_path = os.environ.get("emailMDB")
out_path = os.environ.get("email")

if not mdb_path or not out_path:
    raise ValueError(
        "Environment variables 'emailMDB' and 'email' must be set."
    )

# 2. Read the main Excel file
df = pd.read_excel(mdb_path)

# Ensure 'Last Sent Date' is in datetime format
df["Last Sent Date"] = pd.to_datetime(df["Last Sent Date"], errors="coerce")

# 3. Define filtering conditions
now = pd.Timestamp(datetime.now().date())
seven_days_ago = now - pd.Timedelta(days=7)

# Condition 1: Last Sent Date was more than 7 days ago (or never sent/NaT)
cond_date = (df["Last Sent Date"] < seven_days_ago) | (
    df["Last Sent Date"].isna()
)

# Condition 2: Email ID Status is not 'DND'
cond_status = (
    df["Email ID Status"].astype(str).str.strip().str.upper() != "DND"
)

# Eligible rows match either condition
eligible_df = df[cond_date & cond_status].copy()

# 4. Pick up to 400 random entries
sample_size = min(400, len(eligible_df))
subset_df = eligible_df.sample(n=sample_size, random_state=None).copy()

# 6. Save to the new Excel file path
subset_df.to_excel(out_path, index=False)
print(f"Successfully exported {sample_size} records to: {out_path}")

Successfully exported 400 records to: F:\Chimney Work\Marketing\LeadGen\Data Architecture\3 - Gold\Email list\Email list.xlsx
